In [ ]:
# Import required libraries
import os
import kagglehub
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    transforms.Resize((28, 28)), # TODO: Resize to 28x28
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    transforms.ToTensor(),# TODO: Convert to Tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here

# Load MNIST dataset
# train_dataset = EMNIST(root="./datasets", train=True, transform=transform, download=True)
# train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)


# Create DataLoader
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
print(f"Training samples: {len(train_dataset)}")


# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# # test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

# print(f"Train samples: {len(train_dataset)}")
# print(f"Test samples: {len(test_dataset)}")

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s
import torch
import torchvision.models as models

# Write your code here
# Load pretrained EfficientNetV2-S model
device = "cuda" if torch.cuda.is_available() else "cpu"
efficientnet = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
efficientnet.eval().to(device)



In [ ]:
efficientnet.classifier[1].in_features

In [ ]:

bias= True
num_classes = 26  # 26 letter classes
in_features = efficientnet.classifier[1].in_features  # Input features for predictor

# Replace final layer with new predictor
efficientnet.classifier = efficientnet.classifier[1](in_features, num_classes, bias) # i want to pass required prams

In [ ]:
for param in efficientnet.features.parameters(): #freeze all prams
    param.requires_grad = False

efficientnet.classifier = efficientnet.classifier.requires_grad_(True) #un freeze the head


In [ ]:
# Move efficientnet to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
efficientnet.to(device)

efficientnet

In [ ]:
# Write your code here
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(models.classifier.parameters(), lr=0.001, momentum=0.9)

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # Zero the parameter gradients

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy


def validate_epoch(model, dataloader, criterion, device):
  model.eval() # Set the model to evaluation mode
  running_loss = 0.0
  correct_predictions = 0
  total_samples = 0

  with torch.no_grad(): # Disable gradient calculation
      for inputs, labels in dataloader:
          inputs, labels = inputs.to(device), labels.to(device)

          outputs = model(inputs)
          loss = criterion(outputs, labels)

          running_loss += loss.item() * inputs.size(0)
          _, predicted = torch.max(outputs.data, 1)
          total_samples += labels.size(0)
          correct_predictions += (predicted == labels).sum().item()

  epoch_loss = running_loss / total_samples
  epoch_accuracy = correct_predictions / total_samples
  return epoch_loss, epoch_accuracy


In [ ]:
# Write your code here
num_epochs = 4

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

print("Starting Training...")
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(models, criterion, optimizer, device)
    val_loss, val_acc = validate_epoch(models, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

print("Finished Training.")

In [ ]:
# Write your code here
